In [1]:
import pandas as pd
import numpy as np
import glob
import os
from google.colab import drive

# 1. Conexión y Rutas
drive.mount('/content/drive')
path = '/content/drive/MyDrive/bt'
all_files = glob.glob(os.path.join(path, "*.csv"))

# 2. Configuración de Memoria
CHUNK_SIZE = 10000  # Procesar 10k filas a la vez
d_types = {
    'Price': 'float32', 'Node_C1_Micro': 'float32', 'Node_C5_Macro': 'float32',
    'Upsilon_Tension': 'float32', 'Energy_K': 'float32', 'Tick_Speed': 'float32',
    'Confluence_Score': 'int8'
}

success_data = [] # Para Spikes > 5 pips
failure_data = [] # Para Saturación < 0.20 sin movimiento

print(f"📡 Iniciando Minería de Contraste en {len(all_files)} archivos...")

for filename in all_files:
    # Usamos el iterador de chunks para no saturar RAM
    reader = pd.read_csv(filename, dtype=d_types, chunksize=CHUNK_SIZE)

    for chunk in reader:
        chunk['Price_Diff'] = chunk['Price'].diff()

        # --- CASO A: ÉXITO (Ignición Real) ---
        success_mask = (chunk['Price_Diff'] > 5.0)
        if success_mask.any():
            success_data.append(chunk[success_mask].copy())

        # --- CASO B: FALLO (Saturación Estéril) ---
        # Definición: Upsilon bajo pero el precio no se movió más de 0.5 pips
        failure_mask = (chunk['Upsilon_Tension'] < 0.20) & (chunk['Price_Diff'].abs() < 0.5)
        if failure_mask.any():
            # Tomamos una muestra representativa del fallo para no llenar la RAM
            failure_data.append(chunk[failure_mask].sample(frac=0.1).copy())

    print(f"✅ Archivo procesado: {os.path.basename(filename)}")

# 3. Consolidación de Matrices
df_success = pd.concat(success_data, ignore_index=True)
df_failure = pd.concat(failure_data, ignore_index=True)

# 4. Análisis de Contraste (Diferencia de Firmas)
print("\n--- 🔱 MATRIZ DE CONTRASTE ARC ---")
print("Métrica | Firma ÉXITO | Firma FALLO")
metrics = ['Energy_K', 'Tick_Speed', 'Node_C1_Micro', 'Node_C5_Macro']

for m in metrics:
    print(f"{m:15} | {df_success[m].mean():10.2f} | {df_failure[m].mean():10.2f}")

# Guardar resultados para el EA v43.0
df_success.to_csv('/content/drive/MyDrive/ARC_SIGNATURE_SUCCESS.csv', index=False)
df_failure.to_csv('/content/drive/MyDrive/ARC_SIGNATURE_FAILURE.csv', index=False)

print("\n💾 Archivos de firma guardados. Listos para programar filtros de bloqueo.")

Mounted at /content/drive
📡 Iniciando Minería de Contraste en 29 archivos...
✅ Archivo procesado: ARC_GOLD_DATA_Boom 1000 Index_690000.csv
✅ Archivo procesado: ARC_GOLD_DATA_Boom 1000 Index_695625.csv
✅ Archivo procesado: ARC_GOLD_DATA_Boom 1000 Index_696937.csv
✅ Archivo procesado: ARC_GOLD_DATA_Boom 1000 Index_698687.csv
✅ Archivo procesado: ARC_GOLD_DATA_Boom 1000 Index_702671.csv
✅ Archivo procesado: ARC_GOLD_DATA_Boom 1000 Index_707156.csv
✅ Archivo procesado: ARC_GOLD_DATA_Boom 1000 Index_712468.csv
✅ Archivo procesado: ARC_GOLD_DATA_Boom 1000 Index_845375.csv
✅ Archivo procesado: ARC_GOLD_DATA_Boom 1000 Index_863875.csv
✅ Archivo procesado: ARC_GOLD_DATA_Boom 1000 Index_880218.csv
✅ Archivo procesado: ARC_GOLD_DATA_Boom 1000 Index_897187.csv
✅ Archivo procesado: ARC_GOLD_DATA_Boom 1000 Index_911140.csv
✅ Archivo procesado: ARC_GOLD_DATA_Boom 1000 Index_929203.csv
✅ Archivo procesado: ARC_GOLD_DATA_Boom 1000 Index_1000921.csv
✅ Archivo procesado: ARC_GOLD_DATA_Boom 1000 Index_100

In [4]:
import pandas as pd
import numpy as np
import glob
import os
from google.colab import drive

# 1. Conexión y Rutas
drive.mount('/content/drive')
path = '/content/drive/MyDrive/bt'
all_files = glob.glob(os.path.join(path, "*.csv"))

# 2. Configuración de Memoria
CHUNK_SIZE = 10000  # Procesar 10k filas a la vez
d_types = {
    'Price': 'float32', 'Node_C1_Micro': 'float32', 'Node_C5_Macro': 'float32',
    'Upsilon_Tension': 'float32', 'Energy_K': 'float32', 'Tick_Speed': 'float32',
    'Confluence_Score': 'int8'
}

success_data = [] # Para Spikes > 5 pips
failure_data = [] # Para Saturación < 0.20 sin movimiento

print(f"📡 Iniciando Minería de Contraste en {len(all_files)} archivos...")

for filename in all_files:
    # Usamos el iterador de chunks para no saturar RAM
    reader = pd.read_csv(filename, dtype=d_types, chunksize=CHUNK_SIZE)

    for chunk in reader:
        chunk['Price_Diff'] = chunk['Price'].diff()

        # --- CASO A: ÉXITO (Ignición Real) ---
        success_mask = (chunk['Price_Diff'] > 5.0)
        if success_mask.any():
            success_data.append(chunk[success_mask].copy())

        # --- CASO B: FALLO (Saturación Estéril) ---
        # Definición: Upsilon bajo pero el precio no se movió más de 0.5 pips
        failure_mask = (chunk['Upsilon_Tension'] < 0.20) & (chunk['Price_Diff'].abs() < 0.5)
        if failure_mask.any():
            # Tomamos una muestra representativa del fallo para no llenar la RAM
            failure_data.append(chunk[failure_mask].sample(frac=0.1).copy())

    print(f"✅ Archivo procesado: {os.path.basename(filename)}")

# 3. Consolidación de Matrices
df_success = pd.concat(success_data, ignore_index=True)
df_failure = pd.concat(failure_data, ignore_index=True)

# 4. Análisis de Contraste (Diferencia de Firmas)
print("\n--- 🔱 MATRIZ DE CONTRASTE ARC ---")
print("Métrica | Firma ÉXITO | Firma FALLO")
metrics = ['Energy_K', 'Tick_Speed', 'Node_C1_Micro', 'Node_C5_Macro']

for m in metrics:
    print(f"{m:15} | {df_success[m].mean():10.2f} | {df_failure[m].mean():10.2f}")

# Guardar resultados para el EA v43.0
df_success.to_csv('/content/drive/MyDrive/ARC_SIGNATURE_SUCCESS.csv', index=False)
df_failure.to_csv('/content/drive/MyDrive/ARC_SIGNATURE_FAILURE.csv', index=False)

print("\n💾 Archivos de firma guardados. Listos para programar filtros de bloqueo.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📡 Iniciando Minería de Contraste en 29 archivos...
✅ Archivo procesado: ARC_GOLD_DATA_Boom 1000 Index_690000.csv
✅ Archivo procesado: ARC_GOLD_DATA_Boom 1000 Index_695625.csv
✅ Archivo procesado: ARC_GOLD_DATA_Boom 1000 Index_696937.csv
✅ Archivo procesado: ARC_GOLD_DATA_Boom 1000 Index_698687.csv
✅ Archivo procesado: ARC_GOLD_DATA_Boom 1000 Index_702671.csv
✅ Archivo procesado: ARC_GOLD_DATA_Boom 1000 Index_707156.csv
✅ Archivo procesado: ARC_GOLD_DATA_Boom 1000 Index_712468.csv
✅ Archivo procesado: ARC_GOLD_DATA_Boom 1000 Index_845375.csv
✅ Archivo procesado: ARC_GOLD_DATA_Boom 1000 Index_863875.csv
✅ Archivo procesado: ARC_GOLD_DATA_Boom 1000 Index_880218.csv
✅ Archivo procesado: ARC_GOLD_DATA_Boom 1000 Index_897187.csv
✅ Archivo procesado: ARC_GOLD_DATA_Boom 1000 Index_911140.csv
✅ Archivo procesado: ARC_GOLD_DATA_Boom 1000 Index_929203.csv
✅ Archivo proc

In [6]:
import pandas as pd
import numpy as np
import glob
import os
from google.colab import drive

# 1. Conexión a Drive
drive.mount('/content/drive')
path = '/content/drive/MyDrive/bt' # Asegúrate de que esta carpeta existe en tu Drive
all_files = glob.glob(os.path.join(path, "*.csv"))

if not all_files:
    print("❌ ERROR: No se encontraron archivos CSV en la ruta. Verifica el nombre de la carpeta.")
else:
    print(f"🚀 Analizando Direccionalidad en {len(all_files)} archivos...")

# 2. Configuración de Memoria
dtypes = {
    'Price': 'float32', 'Node_C1_Micro': 'float32', 'Node_C5_Macro': 'float32',
    'Energy_K': 'float32', 'Tick_Speed': 'float32'
}

results = []

for filename in all_files:
    try:
        # Leemos el archivo por chunks para proteger la RAM
        reader = pd.read_csv(filename, dtype=dtypes, chunksize=20000)

        for chunk in reader:
            # A. Velocidad de Nodos (Tendencia en los últimos 20 ticks)
            chunk['Vel_M2'] = chunk['Node_C1_Micro'].diff(20)
            chunk['Vel_H12'] = chunk['Node_C5_Macro'].diff(20)

            # B. Identificar Spikes
            chunk['Price_Jump'] = chunk['Price'].diff()
            spikes = chunk[chunk['Price_Jump'] > 5.0].copy()

            if not spikes.empty:
                for idx, row in spikes.iterrows():
                    # Definición de Estado
                    state = "CONFLUENCIA" if np.sign(row['Vel_M2']) == np.sign(row['Vel_H12']) else "TIJERA"
                    trend = "UP" if row['Vel_H12'] > 0 else "DOWN"

                    results.append({
                        'Spike_Size': row['Price_Jump'],
                        'Estado_Fractal': state,
                        'Macro_Trend': trend,
                        'Energy': row['Energy_K']
                    })
        print(f"✅ Procesado: {os.path.basename(filename)}")
    except Exception as e:
        print(f"⚠️ Error en {filename}: {e}")

# 3. Informe Final de Inteligencia
if results:
    df_res = pd.DataFrame(results)
    print("\n--- 🔱 INFORME DE DIRECCIONALIDAD ARC ---")
    summary = df_res.groupby(['Estado_Fractal', 'Macro_Trend'])['Spike_Size'].agg(['count', 'mean', 'max'])
    print(summary)

    # Guardar el conocimiento extraído
    df_res.to_csv('/content/drive/MyDrive/ARC_FINAL_KNOWLEDGE.csv', index=False)
    print("\n💾 Conocimiento guardado en: ARC_FINAL_KNOWLEDGE.csv")
else:
    print("❌ No se detectaron spikes en el análisis. Verifica los datos.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 Analizando Direccionalidad en 29 archivos...
✅ Procesado: ARC_GOLD_DATA_Boom 1000 Index_690000.csv
✅ Procesado: ARC_GOLD_DATA_Boom 1000 Index_695625.csv
✅ Procesado: ARC_GOLD_DATA_Boom 1000 Index_696937.csv
✅ Procesado: ARC_GOLD_DATA_Boom 1000 Index_698687.csv
✅ Procesado: ARC_GOLD_DATA_Boom 1000 Index_702671.csv
✅ Procesado: ARC_GOLD_DATA_Boom 1000 Index_707156.csv
✅ Procesado: ARC_GOLD_DATA_Boom 1000 Index_712468.csv
✅ Procesado: ARC_GOLD_DATA_Boom 1000 Index_845375.csv
✅ Procesado: ARC_GOLD_DATA_Boom 1000 Index_863875.csv
✅ Procesado: ARC_GOLD_DATA_Boom 1000 Index_880218.csv
✅ Procesado: ARC_GOLD_DATA_Boom 1000 Index_897187.csv
✅ Procesado: ARC_GOLD_DATA_Boom 1000 Index_911140.csv
✅ Procesado: ARC_GOLD_DATA_Boom 1000 Index_929203.csv
✅ Procesado: ARC_GOLD_DATA_Boom 1000 Index_1000921.csv
✅ Procesado: ARC_GOLD_DATA_Boom 1000 Index_1008546.csv
✅ Procesado: